In [0]:

# CELL — CREATE SCHEMA CONFIGURATION TABLE

from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

# Reuse the existing project storage configuration
STORAGE_ACCOUNT = "nestlesalesdatalake"
CONTAINER = "customer-sales-datalake"

BASE_PATH = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"

# Define the table structure explicitly
schema_config_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("source_system", StringType(), False),
    StructField("schema_ddl", StringType(), True),
    StructField("schema_version", IntegerType(), False),
    StructField("is_active", BooleanType(), False),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

# Create only if the Delta table does not already exist
if not DeltaTable.isDeltaTable(spark, SCHEMA_CONFIG_PATH):

    empty_schema_config_df = spark.createDataFrame(
        [],
        schema_config_schema
    )

    (
        empty_schema_config_df.write
        .format("delta")
        .mode("error")
        .save(SCHEMA_CONFIG_PATH)
    )

    print("Schema configuration Delta table created.")

else:
    print("Schema configuration Delta table already exists.")

# Verify the table
schema_config_df = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
)

schema_config_df.printSchema()
display(schema_config_df)


Schema configuration Delta table already exists.
root
 |-- dataset_name: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- schema_ddl: string (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)



dataset_name,source_system,schema_ddl,schema_version,is_active,created_at,updated_at


In [0]:

# REGISTER SCHEMA — GOOGLE ADS CAMPAIGNS

from pyspark.sql import functions as F
from datetime import datetime, timezone

SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"

DATASET_NAME = "google_ads_campaigns"
SOURCE_SYSTEM = "google_ads"

SCHEMA_DDL = """
customer_id STRING,
campaign_id STRING,
campaign_resource_name STRING,
campaign_name STRING,
campaign_status STRING,
advertising_channel_type STRING,
advertising_channel_sub_type STRING,
campaign_budget_resource_name STRING,
bidding_strategy_type STRING,
start_date STRING,
end_date STRING,
campaign_primary_status STRING,
serving_status STRING,
target_country STRING,
business_brand STRING,
product_category STRING,
business_objective STRING,
source_extract_timestamp STRING
"""

schema_table = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
)

active_exists = (
    schema_table
    .filter(
        (F.col("dataset_name") == DATASET_NAME)
        & (F.col("is_active") == True)
    )
    .limit(1)
    .count()
)

if active_exists:
    print(
        f"An active schema already exists for {DATASET_NAME}. "
        "No row inserted."
    )
else:
    now = datetime.now(timezone.utc).replace(tzinfo=None)

    new_schema_row = spark.createDataFrame(
        [(
            DATASET_NAME,
            SOURCE_SYSTEM,
            SCHEMA_DDL.strip(),
            1,
            True,
            now,
            now
        )],
        schema=schema_table.schema
    )

    (
        new_schema_row.write
        .format("delta")
        .mode("append")
        .save(SCHEMA_CONFIG_PATH)
    )

    print(f"Schema registered for {DATASET_NAME}.")

# Verify the configuration
display(
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
    .filter(F.col("dataset_name") == DATASET_NAME)
)


Schema registered for google_ads_campaigns.


dataset_name,source_system,schema_ddl,schema_version,is_active,created_at,updated_at
google_ads_campaigns,google_ads,"customer_id STRING, campaign_id STRING, campaign_resource_name STRING, campaign_name STRING, campaign_status STRING, advertising_channel_type STRING, advertising_channel_sub_type STRING, campaign_budget_resource_name STRING, bidding_strategy_type STRING, start_date STRING, end_date STRING, campaign_primary_status STRING, serving_status STRING, target_country STRING, business_brand STRING, product_category STRING, business_objective STRING, source_extract_timestamp STRING",1,true,2026-10-29T04:14:08.723Z,2026-10-29T04:14:08.723Z


In [0]:

# VERIFY REGISTERED SCHEMA

from pyspark.sql.types import StructType
from pyspark.sql import functions as F

SCHEMA_CONFIG_PATH = BASE_PATH + "metadata/schema_config/"

DATASET_NAME = "google_ads_campaigns"

rows = (
    spark.read
    .format("delta")
    .load(SCHEMA_CONFIG_PATH)
    .filter(
        (F.col("dataset_name") == DATASET_NAME)
        & (F.col("is_active") == True)
    )
    .select("schema_ddl", "schema_version")
    .limit(2)
    .collect()
)

if len(rows) != 1:
    raise ValueError(
        f"Expected exactly one active schema for {DATASET_NAME}; "
        f"found {len(rows)}."
    )

SCHEMA_DDL = rows[0]["schema_ddl"]
SCHEMA_VERSION = rows[0]["schema_version"]

SOURCE_SCHEMA = StructType.fromDDL(SCHEMA_DDL)

print("Dataset:", DATASET_NAME)
print("Schema version:", SCHEMA_VERSION)
print("Number of fields:", len(SOURCE_SCHEMA.fields))

SOURCE_SCHEMA.simpleString()


Dataset: google_ads_campaigns
Schema version: 1
Number of fields: 18


'struct<customer_id:string,campaign_id:string,campaign_resource_name:string,campaign_name:string,campaign_status:string,advertising_channel_type:string,advertising_channel_sub_type:string,campaign_budget_resource_name:string,bidding_strategy_type:string,start_date:string,end_date:string,campaign_primary_status:string,serving_status:string,target_country:string,business_brand:string,product_category:string,business_objective:string,source_extract_timestamp:string>'

In [0]:
file_path = (
    BASE_PATH
    + "landing/google_ads/campaigns/year=2026/month=10/day=29/"
    + "campaigns_20261029.json"
)

df = (
    spark.read
    .option("multiLine", "true")
    .schema(SOURCE_SCHEMA)
    .json(file_path)
)

print("Record count:", df.count())
print("Column count:", len(df.columns))

df.printSchema()
display(df.limit(10))

Record count: 100
Column count: 18
root
 |-- customer_id: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_resource_name: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_status: string (nullable = true)
 |-- advertising_channel_type: string (nullable = true)
 |-- advertising_channel_sub_type: string (nullable = true)
 |-- campaign_budget_resource_name: string (nullable = true)
 |-- bidding_strategy_type: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- campaign_primary_status: string (nullable = true)
 |-- serving_status: string (nullable = true)
 |-- target_country: string (nullable = true)
 |-- business_brand: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- business_objective: string (nullable = true)
 |-- source_extract_timestamp: string (nullable = true)



customer_id,campaign_id,campaign_resource_name,campaign_name,campaign_status,advertising_channel_type,advertising_channel_sub_type,campaign_budget_resource_name,bidding_strategy_type,start_date,end_date,campaign_primary_status,serving_status,target_country,business_brand,product_category,business_objective,source_extract_timestamp
1234500000,987650001,customers/1234500000/campaigns/987650001,IN_Nescafe_Search_001,ENABLED,SEARCH,VIDEO_ACTION,customers/1234500000/campaignBudgets/555001,MAXIMIZE_CONVERSIONS,2026-01-04,2026-04-05,PAUSED,NOT_SERVING,IN,Nescafe,Coffee,Brand Awareness,2026-10-29T03:31:00Z
1234500000,987650002,customers/1234500000/campaigns/987650002,IN_KitKat_Display_002,PAUSED,DISPLAY,SMART,customers/1234500000/campaignBudgets/555002,MAXIMIZE_CLICKS,2026-01-07,2026-04-09,LIMITED,UNKNOWN,US,KitKat,Chocolate,Sales,2026-10-29T03:32:00Z
1234500000,987650003,customers/1234500000/campaigns/987650003,IN_Maggi_Video_003,REMOVED,VIDEO,null,customers/1234500000/campaignBudgets/555003,TARGET_CPA,2026-01-10,2026-04-13,ELIGIBLE,SERVING,GB,Maggi,Instant Foods,Conversions,2026-10-29T03:33:00Z
1234500000,987650004,customers/1234500000/campaigns/987650004,IN_Milo_Shopping_004,ENABLED,SHOPPING,VIDEO_ACTION,customers/1234500000/campaignBudgets/555004,MANUAL_CPC,2026-01-13,null,PAUSED,NOT_SERVING,AU,Milo,Malted Beverages,Reach,2026-10-29T03:34:00Z
1234500000,987650005,customers/1234500000/campaigns/987650005,IN_Cerelac_Performance_Max_005,PAUSED,PERFORMANCE_MAX,SMART,customers/1234500000/campaignBudgets/555005,TARGET_ROAS,2026-01-16,2026-04-21,LIMITED,UNKNOWN,SG,Cerelac,Nutrition,Consideration,2026-10-29T03:35:00Z
1234500000,987650006,customers/1234500000/campaigns/987650006,IN_Nestea_Search_006,REMOVED,SEARCH,null,customers/1234500000/campaignBudgets/555006,MAXIMIZE_CONVERSIONS,2026-01-19,2026-04-25,ELIGIBLE,SERVING,CA,Nestea,Beverages,Traffic,2026-10-29T03:36:00Z
1234500000,987650007,customers/1234500000/campaigns/987650007,IN_EveryDay_Display_007,ENABLED,DISPLAY,VIDEO_ACTION,customers/1234500000/campaignBudgets/555007,MAXIMIZE_CLICKS,2026-01-22,2026-04-29,PAUSED,NOT_SERVING,IN,EveryDay,Dairy,Sales,2026-10-29T03:37:00Z
1234500000,987650008,customers/1234500000/campaigns/987650008,IN_Purina_Video_008,PAUSED,VIDEO,SMART,customers/1234500000/campaignBudgets/555008,TARGET_CPA,2026-01-25,null,LIMITED,UNKNOWN,US,Purina,Pet Care,Conversions,2026-10-29T03:38:00Z
1234500000,987650009,customers/1234500000/campaigns/987650009,IN_Nescafe_Shopping_009,REMOVED,SHOPPING,null,customers/1234500000/campaignBudgets/555009,MANUAL_CPC,2026-01-28,2026-05-07,ELIGIBLE,SERVING,GB,Nescafe,Coffee,Brand Awareness,2026-10-29T03:39:00Z
1234500000,987650010,customers/1234500000/campaigns/987650010,IN_KitKat_Performance_Max_010,ENABLED,PERFORMANCE_MAX,VIDEO_ACTION,customers/1234500000/campaignBudgets/555010,TARGET_ROAS,2026-01-31,2026-05-11,PAUSED,NOT_SERVING,AU,KitKat,Chocolate,Sales,2026-10-29T03:40:00Z
